In [2]:
%run setup.py

OISCurve | currency=EUR | valuation_date=2026-03-24
NSSCurve | currency=EUR | valuation_date=2026-04-20


# Asset Pricing: Interest Rate Swaps

## 1. Basics

An interest rate swap (IRS) is an agreement to exchange a stream of
fixed rate cash flows for floating rate cash flows on a notional amount
over a defined term. No principal is exchanged -- only the net interest
payments.

**Multi-curve framework:**

Post-2008 swap pricing uses two separate curves:
- **OIS curve (ESTR)** -- discounting curve for present value of all cash flows
- **EURIBOR curve** -- projection curve for floating rate cash flows

Before 2008 a single LIBOR curve served both purposes. The basis spread
between OIS and EURIBOR that emerged during the financial crisis made the
single-curve approach inaccurate and the multi-curve framework became
market standard, codified under EMIR for collateralised derivatives.

**Instrument covered:**
- Fixed vs floating EUR interest rate swap
- Par swap rate -- the fixed rate that sets NPV to zero at inception
- NPV of an existing off-market swap
- DV01 and key rate sensitivities
- Fixed and floating leg decomposition

## 2 Building the EURIBOR Projection Curve

In the multi-curve framework two curves are needed to price a swap:

- **OIS curve** -- discounts all cash flows to present value
- **EURIBOR curve** -- projects the future floating rate cash flows

In production the EURIBOR curve is bootstrapped from market quotes:
EURIBOR deposits (1W to 12M), FRAs or EURIBOR futures (short end),
and EURIBOR-indexed swap rates (long end). These quotes are sourced
from Bloomberg or Refinitiv and are not freely available.

Here we construct a stylised EURIBOR curve by adding a constant basis
spread over the OIS curve. This spread -- the OIS-EURIBOR basis --
reflects interbank credit and liquidity risk. It averaged 10-15bps in
normal markets, widened to over 200bps during the 2008 crisis, and
spiked again during COVID. For the current EUR market a spread of
approximately 12bps is representative.

This simplification is standard in academic and regulatory stress
testing contexts where the focus is on methodology rather than exact
market calibration.

In [ ]:
# -----------------------------------------------------------------------
# EURIBOR projection curve -- 3M EURIBOR
# -----------------------------------------------------------------------
# We build a simplified EURIBOR
# curve by adding a spread over OIS -- the OIS-EURIBOR basis spread.


# OIS-EURIBOR 3M basis spread -- historically 10-15bps in normal markets
# widened to 200bps+ during the 2008 crisis and 50bps+ during COVID
euribor_spread_bps = 12  # bps -- approximate current 3M EURIBOR - ESTR spread

# build EURIBOR zero curve as OIS + constant spread
# simplified approach -- production would bootstrap from market quotes
euribor_tenors = [1/12, 2/12, 3/12, 6/12, 9/12, 1, 2, 3, 5, 10, 15]

val_parts      = ois_curve.valuation_date.split("-")
valuation_date = ql.Date(
    int(val_parts[2]), int(val_parts[1]), int(val_parts[0])
)
calendar       = ql.TARGET()
day_count_swap = ql.Actual360()  # EURIBOR convention

ql.Settings.instance().evaluationDate = valuation_date

# EURIBOR pillar dates and rates
euribor_dates = [valuation_date] + [
    valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in euribor_tenors
]
euribor_rates = [
    ois_curve.zero_rate(1/365) / 100 + euribor_spread_bps / 10000
] + [
    ois_curve.zero_rate(t) / 100 + euribor_spread_bps / 10000
    for t in euribor_tenors
]

euribor_zc = ql.ZeroCurve(
    euribor_dates,
    euribor_rates,
    day_count_swap,
    calendar,
    ql.Linear(),
    ql.Continuous
)
euribor_zc.enableExtrapolation()
euribor_handle = ql.YieldTermStructureHandle(euribor_zc)

# OIS discount curve handle
ois_times = [0.0] + euribor_tenors
ois_dates  = [
    valuation_date if t == 0
    else valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in ois_times
]
ois_rates  = [
    ois_curve.zero_rate(1/365) / 100 if t == 0
    else ois_curve.zero_rate(t) / 100
    for t in ois_times
]

ois_zc = ql.ZeroCurve(
    ois_dates, ois_rates, day_count_swap,
    calendar, ql.Linear(), ql.Continuous
)
ois_zc.enableExtrapolation()
ois_handle = ql.YieldTermStructureHandle(ois_zc)

print(f"Valuation date      : {valuation_date}")
print(f"EURIBOR spread      : {euribor_spread_bps}bps over OIS")
print(f"\n{'Tenor':<8} {'OIS rate':>10} {'EURIBOR rate':>14} {'Spread':>8}")
print("-" * 44)
for t in euribor_tenors:
    ois_r     = ois_curve.zero_rate(t)
    eur_r     = ois_r + euribor_spread_bps / 100
    label     = f"{int(t*12)}M" if t < 1 else f"{int(t)}Y"
    print(f"{label:<8} {ois_r:>10.4f}% {eur_r:>13.4f}% {euribor_spread_bps:>7}bps")

Valuation date      : March 24th, 2026
EURIBOR spread      : 12bps over OIS

Tenor      OIS rate   EURIBOR rate   Spread
--------------------------------------------
1M           1.9366%        2.0566%      12bps
2M           1.9470%        2.0670%      12bps
3M           1.9786%        2.0986%      12bps
6M           2.0521%        2.1721%      12bps
9M           1.9842%        2.1042%      12bps
1Y           2.0026%        2.1226%      12bps
2Y           2.1312%        2.2512%      12bps
3Y           2.1857%        2.3057%      12bps
5Y           2.3096%        2.4296%      12bps
10Y          2.6167%        2.7367%      12bps
15Y          2.9652%        3.0852%      12bps


#### Note: Interbank rates

These are all regional IBOR (Interbank Offered Rate) for EU member states that are not in the eurozone or non-EU European countries. They follow the same panel-bank methodology as EURIBOR but for their local currencies.

* PRIBOR -- Prague Interbank Offered Rate -- Czech Republic (CZK)
* WIBOR -- Warsaw Interbank Offered Rate -- Poland (PLN)
* BUBOR -- Budapest Interbank Offered Rate -- Hungary (HUF)
* ROBOR -- Romanian Interbank Offered Rate -- Romania (RON)
* NIBOR -- Norwegian Interbank Offered Rate -- Norway (NOK)
* STIBOR -- Stockholm Interbank Offered Rate -- Sweden (SEK)
* CIBOR -- Copenhagen Interbank Offered Rate -- Denmark (DKK)

## 3 Lifecycle of an IRS

This is tha plan fo rthe rest of this notebok, following the lifecycle of an IRS.

1. **Price discovery** -- build a QuantLib `VanillaSwap` with a
   placeholder fixed rate and call `fairRate()` to find the par swap
   rate -- the rate at which the swap has zero NPV today.

2. **Inception** -- build a second swap at the par rate and confirm
   NPV is exactly zero. This represents the swap as traded on day one.

3. **Mark-to-market** -- advance the valuation date by 2 months and
   shift the OIS curve up by 50bps to simulate a rate rise. Reprice
   the swap and show the NPV gain for the pay-fixed counterparty.

4. **Risk** -- compute DV01, key rate sensitivities, and decompose
   the NPV into fixed and floating leg contributions.

## 4 Price Discovery -- Par Rate

We price a EUR 10M notional 5-year pay-fixed receiver-floating swap.
The fixed leg pays an annual coupon at the par swap rate. The floating
leg receives 3M EURIBOR quarterly, projected from the EURIBOR curve
and discounted at OIS.

**Par swap rate** -- the fixed rate that sets the NPV of the swap to
zero at inception. A swap entered at the par rate has zero fair value
on day one.

**NPV** -- for an existing off-market swap (fixed rate different from
par), the NPV reflects the mark-to-market gain or loss.

In [5]:
# -----------------------------------------------------------------------
# section 4 -- price discovery
# build swap with placeholder rate and extract par rate
# -----------------------------------------------------------------------

notional        = 10_000_000                          # EUR 10M notional
swap_maturity   = 5                                   # 5Y tenor
settlement_days = 2
fixed_freq      = ql.Annual                           # fixed leg pays annually
float_freq      = ql.Quarterly                        # floating resets and pays quarterly
fixed_dc        = ql.Thirty360(ql.Thirty360.BondBasis)# 30/360 -- fixed leg convention
float_dc        = ql.Actual360()                      # ACT/360 -- EURIBOR convention

# swap dates
start_date = calendar.advance(valuation_date, settlement_days, ql.Days)
end_date   = calendar.advance(start_date, ql.Period(swap_maturity, ql.Years))

print(f"Swap start : {start_date}")
print(f"Swap end   : {end_date}")

# schedules
fixed_schedule = ql.Schedule(
    start_date,
    end_date,
    ql.Period(fixed_freq),
    calendar,
    ql.ModifiedFollowing,
    ql.ModifiedFollowing,
    ql.DateGeneration.Forward,
    False
)

float_schedule = ql.Schedule(
    start_date,
    end_date,
    ql.Period(float_freq),
    calendar,
    ql.ModifiedFollowing,
    ql.ModifiedFollowing,
    ql.DateGeneration.Forward,
    False
)

# EURIBOR 3M index
euribor3m = ql.Euribor3M(euribor_handle)

# build swap with placeholder rate -- fairRate() is independent of this
swap_pricer = ql.VanillaSwap(
    ql.VanillaSwap.Payer,  # pay fixed, receive floating
    notional,
    fixed_schedule,
    0.0,                   # placeholder -- only used for NPV, not fairRate()
    fixed_dc,
    float_schedule,
    euribor3m,
    0.0,                   # zero spread over EURIBOR
    float_dc,
)
swap_pricer.setPricingEngine(ql.DiscountingSwapEngine(ois_handle))

# extract par rate
par_rate = swap_pricer.fairRate() * 100

print(f"\nPar swap rate : {par_rate:.4f}%")
print(f"OIS 5Y rate   : {ois_curve.zero_rate(5.0):.4f}%")
print(f"Spread        : {par_rate - ois_curve.zero_rate(5.0):.4f}% "
      f"(EURIBOR basis embedded in par rate)")

Swap start : March 26th, 2026
Swap end   : March 26th, 2031

Par swap rate : 2.4889%
OIS 5Y rate   : 2.3096%
Spread        : 0.1793% (EURIBOR basis embedded in par rate)


## 5 Inception -- Swap at Par

The swap is traded at the par rate -- NPV is zero by construction.
The fixed leg PV and floating leg PV are equal and opposite.
No cash changes hands at inception other than any upfront fee.

In [6]:
# -----------------------------------------------------------------------
# section 5 -- inception -- swap at par rate
# -----------------------------------------------------------------------

# swap at par rate 
swap_inception = ql.VanillaSwap(
    ql.VanillaSwap.Payer,  # pay fixed, receive floating
    notional,              # EUR 10M
    fixed_schedule,        # annual fixed leg schedule
    par_rate / 100,        # par rate from section 4 -- sets NPV to zero
    fixed_dc,              # 30/360 -- fixed leg day count
    float_schedule,        # quarterly floating leg schedule
    euribor3m,             # Euribor3M(handle): rates + conventions + tenor + calendar
    0.0,                   # zero spread over EURIBOR -- vanilla swap
    float_dc,              # ACT/360 -- EURIBOR day count
)
swap_inception.setPricingEngine(ql.DiscountingSwapEngine(ois_handle))

npv_inception       = swap_inception.NPV()
fixed_npv_inception = swap_inception.fixedLegNPV()
float_npv_inception = swap_inception.floatingLegNPV()

print(f"Swap at inception -- par rate {par_rate:.4f}%")
print(f"\n{'NPV':<25} : EUR {npv_inception:>12,.2f}  (should be ~0)")
print(f"{'Fixed leg NPV':<25} : EUR {fixed_npv_inception:>12,.2f}")
print(f"{'Floating leg NPV':<25} : EUR {float_npv_inception:>12,.2f}")
print(f"\nFixed + Floating    : EUR {fixed_npv_inception + float_npv_inception:>12,.2f}")

# cash flow table
print(f"\nFixed leg cash flows:")
print(f"{'Date':<25} {'Amount':>12}")
print("-" * 39)
for cf in swap_inception.fixedLeg():
    print(f"{str(cf.date()):<25} {cf.amount():>12,.2f}")

print(f"\nFloating leg cash flows (projected):")
print(f"{'Date':<25} {'Amount':>12}")
print("-" * 39)
for cf in swap_inception.floatingLeg():
    print(f"{str(cf.date()):<25} {cf.amount():>12,.2f}")

Swap at inception -- par rate 2.4889%

NPV                       : EUR         0.00  (should be ~0)
Fixed leg NPV             : EUR -1,163,532.83
Floating leg NPV          : EUR 1,163,532.83

Fixed + Floating    : EUR         0.00

Fixed leg cash flows:
Date                            Amount
---------------------------------------
March 30th, 2027            251,658.91
March 27th, 2028            246,819.32
March 26th, 2029            248,202.06
March 26th, 2030            248,893.43
March 26th, 2031            248,893.43

Floating leg cash flows (projected):
Date                            Amount
---------------------------------------
June 26th, 2026              53,863.56
September 28th, 2026         58,505.58
December 28th, 2026          50,100.66
March 30th, 2027             55,951.99
June 28th, 2027              57,340.67
September 27th, 2027         59,600.12
December 27th, 2027          61,229.83
March 27th, 2028             62,693.39
June 26th, 2028              60,214.14
Sept

## 6 Mark-to-Market -- Time and Rate Shift

Two months have passed since inception. The ECB has raised rates
by 50bps. We advance the valuation date and rebuild the OIS curve
with a parallel shift to reprice the swap.

The pay-fixed counterparty benefits -- the fixed rate locked at
inception is now below the new market par rate. The swap has
positive NPV for the pay-fixed counterparty.

In [7]:
# -----------------------------------------------------------------------
# section 6 -- mark-to-market after 2 months and +50bps rate shift
# 2 months chosen deliberately -- no cash flows have occurred yet
# first floating payment at 3M, first fixed payment at 12M
# -----------------------------------------------------------------------

# advance valuation date by 2 months
new_valuation_date = calendar.advance(valuation_date, ql.Period(2, ql.Months))
ql.Settings.instance().evaluationDate = new_valuation_date

print(f"Original valuation date : {valuation_date}")
print(f"New valuation date      : {new_valuation_date}")
print(f"No cash flows have occurred -- first floating pmt at 3M, fixed at 12M")

# rebuild OIS curve with +50bps parallel shift
rate_shift    = 0.0050  # 50bps

ois_times_mtm = [0.0, 1/12, 2/12, 3/12, 6/12, 9/12, 1, 2, 3, 5, 10, 15]
ois_dates_mtm = [
    new_valuation_date if t == 0
    else new_valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in ois_times_mtm
]
ois_rates_mtm = [
    ois_curve.zero_rate(1/365) / 100 + rate_shift if t == 0
    else ois_curve.zero_rate(t) / 100 + rate_shift
    for t in ois_times_mtm
]

ois_zc_mtm = ql.ZeroCurve(
    ois_dates_mtm, ois_rates_mtm, day_count_swap,
    calendar, ql.Linear(), ql.Continuous
)
ois_zc_mtm.enableExtrapolation()
ois_handle_mtm = ql.YieldTermStructureHandle(ois_zc_mtm)

# rebuild EURIBOR curve with same shift
euribor_dates_mtm = [
    new_valuation_date if t == 0
    else new_valuation_date + ql.Period(max(1, int(t * 365)), ql.Days)
    for t in ois_times_mtm
]
euribor_rates_mtm = [
    ois_curve.zero_rate(1/365) / 100 + euribor_spread_bps / 10000 + rate_shift if t == 0
    else ois_curve.zero_rate(t) / 100 + euribor_spread_bps / 10000 + rate_shift
    for t in ois_times_mtm
]

euribor_zc_mtm = ql.ZeroCurve(
    euribor_dates_mtm, euribor_rates_mtm, day_count_swap,
    calendar, ql.Linear(), ql.Continuous
)
euribor_zc_mtm.enableExtrapolation()
euribor_handle_mtm = ql.YieldTermStructureHandle(euribor_zc_mtm)

# reprice the swap at new valuation date and rates
euribor3m_mtm = ql.Euribor3M(euribor_handle_mtm)

swap_mtm = ql.VanillaSwap(
    ql.VanillaSwap.Payer,  # pay fixed, receive floating
    notional,              # EUR 10M
    fixed_schedule,        # same schedule as inception
    par_rate / 100,        # same contractual fixed rate
    fixed_dc,              # 30/360
    float_schedule,        # same schedule as inception
    euribor3m_mtm,         # new EURIBOR index with shifted curve
    0.0,                   # zero spread
    float_dc,              # ACT/360
)
swap_mtm.setPricingEngine(ql.DiscountingSwapEngine(ois_handle_mtm))

npv_mtm       = swap_mtm.NPV()
par_rate_mtm  = swap_mtm.fairRate() * 100
fixed_npv_mtm = swap_mtm.fixedLegNPV()
float_npv_mtm = swap_mtm.floatingLegNPV()

print(f"\nMark-to-market -- 2 months later, rates +{rate_shift*10000:.0f}bps")
print(f"\n{'New par rate':<25} : {par_rate_mtm:.4f}%")
print(f"{'Original par rate':<25} : {par_rate:.4f}%")
print(f"{'Rate change':<25} : {par_rate_mtm - par_rate:.4f}%")
print(f"\n{'NPV at inception':<25} : EUR {0:>12,.2f}")
print(f"{'NPV after 2M +50bps':<25} : EUR {npv_mtm:>12,.2f}")
print(f"{'NPV change':<25} : EUR {npv_mtm:>12,.2f}")
print(f"\n{'Fixed leg NPV':<25} : EUR {fixed_npv_mtm:>12,.2f}")
print(f"{'Floating leg NPV':<25} : EUR {float_npv_mtm:>12,.2f}")
print(f"\nPay-fixed counterparty {'gains' if npv_mtm > 0 else 'loses'} "
      f"EUR {abs(npv_mtm):,.2f} after rates rise {rate_shift*10000:.0f}bps")

Original valuation date : March 24th, 2026
New valuation date      : May 25th, 2026
No cash flows have occurred -- first floating pmt at 3M, fixed at 12M


RuntimeError: 46th iteration: failed at 2nd alive instrument, pillar July 27th, 2026, maturity July 27th, 2026, reference date March 24th, 2026: root not bracketed: f[0.201189,0.557362] -> [-2.586160e+01,-4.207549e-02]